# 05 — Análise temporal e estacionariedade

Este notebook analisa, em frequência mensal, as duas séries nacionais usadas no estudo **Mudanças Climáticas e Inadimplência: uma análise do impacto de desastres naturais no sistema de crédito brasileiro**:

- taxa nacional de inadimplência de pessoas físicas;
- quantidade total de desastres naturais.

A sequência segue uma linha temporal de diagnóstico: série original, decomposição clássica, decomposição STL, sazonalidade, estabilidade da variância, testes de estacionariedade, comparação das transformações e inspeção da ACF/PACF. Ao final, são exportadas apenas as transformações selecionadas e alinhadas para o notebook `06_causalidade_granger.ipynb`.

**Período:** janeiro de 2013 a dezembro de 2024.  
**Nível de significância:** 5%.  
**Importante:** estacionariedade é uma condição da modelagem, não evidência de causalidade.

## 1. Bibliotecas, configuração e caminhos

In [1]:
from pathlib import Path
import sys
import warnings
from importlib.util import find_spec

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from IPython.display import display, Markdown
from scipy.stats import kruskal, levene, spearmanr
from statsmodels.api import OLS, add_constant
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import het_arch
from statsmodels.tools.sm_exceptions import InterpolationWarning
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.tsa.stattools import acf, adfuller, kpss

ALPHA = 0.05
PERIODO = 12
MAX_LAGS = 36
INICIO = pd.Timestamp("2013-01-01")
FIM = pd.Timestamp("2024-12-01")
DATAS_ESPERADAS = pd.date_range(INICIO, FIM, freq="MS")

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def encontrar_raiz(inicio):
    inicio = Path(inicio).resolve()
    for pasta in [inicio, *inicio.parents]:
        if (pasta / "data").exists() and (pasta / "notebooks").exists():
            return pasta
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")

ROOT = encontrar_raiz(Path.cwd())
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"
TABLES = ROOT / "outputs" / "tables"
for pasta in (PROCESSED, FIGURES, TABLES):
    pasta.mkdir(parents=True, exist_ok=True)

print("Raiz do projeto:", ROOT)
print("Interpretador:", sys.executable)
print("Python:", sys.version.split()[0])
print("Suporte a Parquet:", "sim" if find_spec("pyarrow") or find_spec("fastparquet") else "não")

Raiz do projeto: /workspace/scratch/94bf83bea5df/repo
Interpretador: /opt/codex/runtimes/codex-primary-runtime/dependencies/python/bin/python
Python: 3.12.14
Suporte a Parquet: sim


## 2. Leitura e validação da base

O Parquet é priorizado; o CSV separado por ponto e vírgula funciona como alternativa. A base deve conter exatamente uma observação por UF e mês, 27 UFs e os 144 meses do período. As colunas analíticas são convertidas explicitamente para `float64`, evitando incompatibilidade entre valores `Decimal` e funções NumPy.

In [2]:
arquivo_parquet = PROCESSED / "df_tcc_2013_2024.parquet"
arquivo_csv = PROCESSED / "df_tcc_2013_2024.csv"

if arquivo_parquet.exists() and (find_spec("pyarrow") or find_spec("fastparquet")):
    df = pd.read_parquet(arquivo_parquet)
    origem = arquivo_parquet
elif arquivo_csv.exists():
    df = pd.read_csv(arquivo_csv, sep=";")
    origem = arquivo_csv
elif arquivo_parquet.exists():
    raise ImportError("Instale pyarrow no kernel atual: python -m pip install pyarrow")
else:
    raise FileNotFoundError("Coloque df_tcc_2013_2024.parquet ou .csv em data/processed/.")

obrigatorias = {
    "uf", "data_base", "carteira_ativa_total",
    "carteira_inadimplencia_total", "total_desastres"
}
faltantes = sorted(obrigatorias - set(df.columns))
if faltantes:
    raise ValueError(f"Colunas obrigatórias ausentes: {faltantes}")

df["data_base"] = pd.to_datetime(df["data_base"], errors="raise").dt.to_period("M").dt.to_timestamp()
df["uf"] = df["uf"].astype("string").str.strip().str.upper()
for coluna in ["carteira_ativa_total", "carteira_inadimplencia_total", "total_desastres"]:
    df[coluna] = pd.to_numeric(df[coluna], errors="raise").astype("float64")
df = df.loc[df["data_base"].between(INICIO, FIM)].sort_values(["data_base", "uf"]).reset_index(drop=True)

validacao = pd.DataFrame({
    "Verificação": ["Linhas", "UFs", "Meses", "Duplicidades UF × mês", "Nulos essenciais", "Infinitos"],
    "Observado": [
        len(df), df["uf"].nunique(), df["data_base"].nunique(),
        int(df.duplicated(["uf", "data_base"]).sum()),
        int(df[list(obrigatorias)].isna().sum().sum()),
        int(np.isinf(df[["carteira_ativa_total", "carteira_inadimplencia_total", "total_desastres"]].to_numpy(dtype="float64")).sum())
    ],
    "Esperado": [3888, 27, 144, 0, 0, 0]
})
validacao["Válido"] = validacao["Observado"] == validacao["Esperado"]
display(validacao)
if not validacao["Válido"].all():
    raise ValueError("A base não atende à estrutura UF × mês esperada.")
print("Arquivo lido:", origem)

Arquivo lido: /workspace/scratch/94bf83bea5df/repo/data/processed/df_tcc_2013_2024.parquet


,Verificação,Observado,Esperado,Válido
0,Linhas,3888,3888,True
1,UFs,27,27,True
2,Meses,144,144,True
3,Duplicidades UF × mês,0,0,True
4,Nulos essenciais,0,0,True
5,Infinitos,0,0,True


## 3. Construção das séries nacionais

A taxa nacional não é uma média simples das taxas estaduais. Ela é reconstruída mensalmente pela razão entre a soma da carteira inadimplente e a soma da carteira ativa:

\[
\text{Inadimplência}_t = 100\times
\frac{\sum_{UF}\text{Carteira inadimplente}_{UF,t}}
{\sum_{UF}\text{Carteira ativa}_{UF,t}}.
\]

Os desastres são somados entre as UFs. Não são analisadas séries separadas por grupo de desastre.

In [3]:
series = (
    df.groupby("data_base")
      .agg(
          carteira_ativa=("carteira_ativa_total", "sum"),
          carteira_inadimplente=("carteira_inadimplencia_total", "sum"),
          total_desastres=("total_desastres", "sum")
      )
      .sort_index()
      .reindex(DATAS_ESPERADAS)
)
series["taxa_inadimplencia"] = 100 * series["carteira_inadimplente"] / series["carteira_ativa"]
series = series[["taxa_inadimplencia", "total_desastres"]].astype("float64")
series.index.name = "data"

if not series.index.equals(DATAS_ESPERADAS):
    raise ValueError("O índice mensal não está completo ou ordenado.")
if series.isna().any().any() or np.isinf(series.to_numpy(dtype="float64")).any():
    raise ValueError("As séries nacionais contêm nulos ou infinitos.")

resumo = series.agg(["count", "mean", "std", "min", "median", "max"]).T
resumo.index = ["Taxa de inadimplência (%)", "Total de desastres"]
display(resumo.rename(columns={"count":"n", "mean":"Média", "std":"Desvio-padrão", "min":"Mínimo", "median":"Mediana", "max":"Máximo"}))

,n,Média,Desvio-padrão,Mínimo,Mediana,Máximo
Taxa de inadimplência (%),144.0000,3.8425,0.4576,2.8480,3.8398,5.0324
Total de desastres,144.0000,280.1319,188.7564,48.0000,235.0000,"1,408.0000"


## 4. Evolução temporal das séries

In [4]:
rotulos = {
    "taxa_inadimplencia": ("Taxa nacional de inadimplência", "Taxa (%)", "#1f4e79"),
    "total_desastres": ("Quantidade nacional de desastres naturais", "Ocorrências", "#b3541e")
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, coluna in zip(axes, series.columns):
    titulo, ylabel, cor = rotulos[coluna]
    ax.plot(series.index, series[coluna], color=cor, linewidth=1.5, label="Série mensal")
    ax.plot(series.index, series[coluna].rolling(12).mean(), color="black", linewidth=2, label="Média móvel (12 meses)")
    ax.set_title(titulo)
    ax.set_ylabel(ylabel)
    ax.legend(loc="best")
axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.suptitle("Evolução mensal das séries nacionais — 2013 a 2024", y=1.01, fontsize=15)
fig.tight_layout()
fig.savefig(FIGURES / "05_series_nacionais_linha_temporal.png", dpi=180, bbox_inches="tight")
plt.show()

<Figure>

## 5. Decomposição clássica aditiva

A decomposição clássica separa a série em tendência, sazonalidade fixa e resíduo. Ela é apresentada primeiro para manter a leitura cronológica do diagnóstico. Como sua sazonalidade se repete com a mesma forma em todos os anos e a tendência usa médias móveis, a STL robusta será usada em seguida como diagnóstico principal.

In [5]:
decomp_classica = {}
for coluna in series.columns:
    titulo, ylabel, _ = rotulos[coluna]
    resultado = seasonal_decompose(series[coluna], model="additive", period=12, extrapolate_trend="freq")
    decomp_classica[coluna] = resultado
    fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
    componentes = [(series[coluna], "Observada"), (resultado.trend, "Tendência"), (resultado.seasonal, "Sazonal"), (resultado.resid, "Resíduo")]
    for ax, (componente, nome) in zip(axes, componentes):
        ax.plot(componente.index, componente, color=rotulos[coluna][2], linewidth=1.2)
        ax.set_ylabel(nome)
    axes[0].set_title(f"Decomposição clássica aditiva — {titulo}")
    axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.tight_layout()
    fig.savefig(FIGURES / f"05_{coluna}_decomposicao_classica.png", dpi=180, bbox_inches="tight")
    plt.show()

<Figure>

<Figure>

cell_10:4: FutureWarning: `extrapolate_trend='freq'` is deprecated and will be removed in 0.16, use `extrapolate_trend='period'` instead.


## 6. Decomposição STL robusta

A STL permite que o padrão sazonal se ajuste gradualmente ao longo do tempo. A opção `robust=True` reduz a influência de observações extremas, importante especialmente para os desastres. As forças de tendência e sazonalidade variam entre 0 e 1; valores próximos de 1 indicam componente relativamente forte.

In [6]:
stl_resultados = {}
forcas = []
for coluna in series.columns:
    titulo, ylabel, cor = rotulos[coluna]
    resultado = STL(series[coluna], period=12, robust=True).fit()
    stl_resultados[coluna] = resultado
    var_resid = np.var(resultado.resid, ddof=1)
    forca_tendencia = max(0, 1 - var_resid / np.var(resultado.trend + resultado.resid, ddof=1))
    forca_sazonal = max(0, 1 - var_resid / np.var(resultado.seasonal + resultado.resid, ddof=1))
    forcas.append({"Série": coluna, "Força da tendência": forca_tendencia, "Força da sazonalidade": forca_sazonal})
    fig, axes = plt.subplots(4, 1, figsize=(14, 9), sharex=True)
    componentes = [(series[coluna], "Observada"), (resultado.trend, "Tendência"), (resultado.seasonal, "Sazonal"), (resultado.resid, "Resíduo")]
    for ax, (componente, nome) in zip(axes, componentes):
        ax.plot(componente.index, componente, color=cor, linewidth=1.2)
        ax.set_ylabel(nome)
    axes[0].set_title(f"Decomposição STL robusta — {titulo}")
    axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.tight_layout()
    fig.savefig(FIGURES / f"05_{coluna}_decomposicao_stl.png", dpi=180, bbox_inches="tight")
    plt.show()

forcas = pd.DataFrame(forcas)
display(forcas)

<Figure>

<Figure>

,Série,Força da tendência,Força da sazonalidade
0,taxa_inadimplencia,0.9068,0.1400
1,total_desastres,0.1803,0.2028


## 7. Sazonalidade

São combinados três elementos: perfil médio por mês, teste de Kruskal–Wallis e ACF nos lags 12, 24 e 36. No teste formal:

- **H0:** as distribuições são iguais entre os meses do ano;
- **H1:** pelo menos um mês apresenta distribuição diferente;
- rejeita-se H0 quando `p-valor < 0,05`.

Esse teste identifica **sazonalidade determinística**, mas não uma raiz unitária sazonal. A existência de médias mensais diferentes não implica, por si só, que seja necessária uma diferença sazonal.

In [7]:
nomes_meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
diagnostico_sazonal = []
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, coluna in zip(axes, series.columns):
    perfil = series[coluna].groupby(series.index.month).mean()
    grupos = [g.to_numpy() for _, g in series[coluna].groupby(series.index.month)]
    kw = kruskal(*grupos)
    acfs = acf(series[coluna], nlags=36, fft=False)
    diagnostico_sazonal.append({
        "Série": coluna, "Kruskal–Wallis": kw.statistic, "p-valor": kw.pvalue,
        "Rejeita H0": kw.pvalue < ALPHA,
        "ACF lag 12": acfs[12], "ACF lag 24": acfs[24], "ACF lag 36": acfs[36]
    })
    ax.bar(nomes_meses, perfil.values, color=rotulos[coluna][2], alpha=.85)
    ax.set_title(rotulos[coluna][0])
    ax.set_ylabel(rotulos[coluna][1])
    ax.tick_params(axis="x", rotation=45)
fig.suptitle("Perfil sazonal médio por mês", y=1.02, fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES / "05_perfis_sazonais.png", dpi=180, bbox_inches="tight")
plt.show()
diagnostico_sazonal = pd.DataFrame(diagnostico_sazonal)
display(diagnostico_sazonal)

<Figure>

,Série,Kruskal–Wallis,p-valor,Rejeita H0,ACF lag 12,ACF lag 24,ACF lag 36
0,taxa_inadimplencia,3.9743,0.9707,False,0.3066,0.0548,0.2242
1,total_desastres,15.4998,0.1607,False,0.2380,0.1364,0.0401


## 8. Estabilidade da variância e heterocedasticidade condicional

Quatro conceitos não devem ser confundidos:

- **variância não constante:** a dispersão muda entre partes do período;
- **heterocedasticidade condicional:** a variância atual depende de choques passados, avaliada aqui pelo ARCH-LM sobre resíduos autorregressivos;
- **volatilidade:** intensidade observada das oscilações;
- **mudança estrutural:** alteração persistente no nível, tendência ou parâmetros da série.

O log não é assumido como solução antecipadamente. Sua utilidade será avaliada comparando a associação entre média e desvio-padrão móveis, a variância nas duas metades da amostra e o teste robusto de Levene.

In [8]:
def diagnosticar_variancia(serie, nome):
    s = pd.Series(serie).dropna().astype(float)
    metade = len(s) // 2
    lev = levene(s.iloc[:metade], s.iloc[metade:], center="median")
    mm = s.rolling(12).mean()
    dp = s.rolling(12).std()
    validos = mm.notna() & dp.notna()
    rho, p_rho = spearmanr(mm[validos], dp[validos])
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            resid = AutoReg(s, lags=12, trend="ct").fit().resid
            lm, p_arch, _, _ = het_arch(resid, nlags=12, ddof=14)
    except Exception:
        lm, p_arch = np.nan, np.nan
    return {
        "Série": nome,
        "Variância 1ª metade": s.iloc[:metade].var(),
        "Variância 2ª metade": s.iloc[metade:].var(),
        "Razão var. (2ª/1ª)": s.iloc[metade:].var() / s.iloc[:metade].var(),
        "Levene p-valor": lev.pvalue,
        "Correlação média × DP móveis": rho,
        "Correlação p-valor": p_rho,
        "ARCH-LM p-valor": p_arch
    }

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
for ax, coluna in zip(axes, series.columns):
    media = series[coluna].rolling(12).mean()
    desvio = series[coluna].rolling(12).std()
    ax.plot(media.index, media, label="Média móvel", color=rotulos[coluna][2])
    ax2 = ax.twinx()
    ax2.plot(desvio.index, desvio, label="Desvio-padrão móvel", color="#6a3d9a", alpha=.8)
    ax.set_title(rotulos[coluna][0])
    ax.set_ylabel("Média móvel")
    ax2.set_ylabel("Desvio-padrão móvel")
fig.suptitle("Média e desvio-padrão móveis de 12 meses", y=1.01, fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES / "05_estabilidade_variancia.png", dpi=180, bbox_inches="tight")
plt.show()

variancia_nivel = pd.DataFrame([
    diagnosticar_variancia(series["taxa_inadimplencia"], "taxa_inadimplencia — nível"),
    diagnosticar_variancia(series["total_desastres"], "total_desastres — nível")
])
display(variancia_nivel)

<Figure>

,Série,Variância 1ª metade,Variância 2ª metade,Razão var. (2ª/1ª),Levene p-valor,Correlação média × DP móveis,Correlação p-valor,ARCH-LM p-valor
0,taxa_inadimplencia — nível,0.1310,0.1611,1.2297,0.3060,-0.0173,0.8434,0.0028
1,total_desastres — nível,"15,736.4687","47,601.9951",3.0249,0.0302,0.7557,0.0000,0.1831


## 9. Testes ADF e KPSS em nível

**ADF**, com constante e defasagens escolhidas por AIC:

- H0: a série possui raiz unitária e é não estacionária;
- H1: a série é estacionária.

**KPSS**, com constante e `nlags="auto"`:

- H0: a série é estacionária em nível;
- H1: a série é não estacionária.

Em ambos os casos, H0 é rejeitada quando o p-valor é inferior a 5%. A leitura conjunta é:

1. ADF rejeita e KPSS não rejeita: evidência convergente de estacionariedade;
2. ADF não rejeita e KPSS rejeita: evidência convergente de não estacionariedade;
3. ambos rejeitam: conflito, possível quebra estrutural ou especificação inadequada;
4. nenhum rejeita: resultado inconclusivo, frequentemente por baixa potência.

In [9]:
def executar_testes(serie, nome, transformacao):
    s = pd.Series(serie).replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    base = {"Série": nome, "Transformação": transformacao, "n": len(s)}
    if len(s) < 36 or s.nunique() <= 1:
        return {**base, "ADF estatística": np.nan, "ADF p-valor": np.nan, "ADF lags": np.nan,
                "KPSS estatística": np.nan, "KPSS p-valor": np.nan, "KPSS lags": np.nan,
                "ADF rejeita H0": False, "KPSS rejeita H0": False, "Conclusão conjunta": "Teste não aplicável"}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", InterpolationWarning)
        adf_r = adfuller(s, regression="c", autolag="AIC")
        kpss_r = kpss(s, regression="c", nlags="auto")
    adf_rej = adf_r[1] < ALPHA
    kpss_rej = kpss_r[1] < ALPHA
    if adf_rej and not kpss_rej:
        conjunta = "Evidência de estacionariedade"
    elif not adf_rej and kpss_rej:
        conjunta = "Evidência de NÃO-estacionariedade"
    else:
        conjunta = "Resultado conflitante ou inconclusivo"
    return {**base, "ADF estatística": adf_r[0], "ADF p-valor": adf_r[1], "ADF lags": adf_r[2],
            "KPSS estatística": kpss_r[0], "KPSS p-valor": kpss_r[1], "KPSS lags": kpss_r[2],
            "ADF rejeita H0": adf_rej, "KPSS rejeita H0": kpss_rej, "Conclusão conjunta": conjunta}

testes_nivel = pd.DataFrame([
    executar_testes(series["taxa_inadimplencia"], "taxa_inadimplencia", "nível"),
    executar_testes(series["total_desastres"], "total_desastres", "nível")
])
display(testes_nivel)

,Série,Transformação,n,ADF estatística,ADF p-valor,ADF lags,KPSS estatística,KPSS p-valor,KPSS lags,ADF rejeita H0,KPSS rejeita H0,Conclusão conjunta
0,taxa_inadimplencia,nível,144,-2.2100,0.2026,13,0.6962,0.0139,8,False,True,Evidência de NÃO-estacionariedade
1,total_desastres,nível,144,-7.9772,0.0000,0,1.1038,0.0100,5,True,True,Resultado conflitante ou inconclusivo


cell_18:10: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
cell_18:11: FutureWarning: kpss currently returns a plain tuple whose length and layout depends on the store argument (and which silently drops `lags` when store=True). In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning a KPSSResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
cell_18:10: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always

## 10. Transformações candidatas

Uma transformação modifica a escala ou a dinâmica da série para tornar suas propriedades mais compatíveis com os pressupostos dos modelos temporais. Ela não deve ser aplicada mecanicamente: cada alternativa responde a um problema específico e altera a interpretação da variável.

Para a inadimplência, que apresenta somente valores positivos, pode-se usar `log(x)`. Para os desastres, por se tratar de uma contagem que poderia assumir valor zero em outra amostra, utiliza-se `log1p(x)=log(1+x)`. Essa forma evita o problema matemático de `log(0)`.

As transformações são identificadas nos gráficos da seguinte maneira:

| Identificação | Transformação | Fórmula | Objetivo e interpretação |
|---|---|---|---|
| Série original | Nível | $x_t$ | Mantém a unidade e a interpretação originais. É preferível quando a série já é estacionária. |
| (a) | Log ou `log1p` | $\log(x_t)$ ou $\log(1+x_t)$ | Pode reduzir a assimetria e estabilizar uma variância que aumenta com o nível. Não garante homocedasticidade. |
| (b) | Primeira diferença | $x_t-x_{t-1}$ | Representa a variação mensal na unidade original e pode remover uma tendência estocástica. |
| (c) | Primeira diferença do log | $\log(x_t)-\log(x_{t-1})$ | Aproxima uma taxa de crescimento mensal, combinando mudança de escala e diferenciação. |
| (d) | Diferença sazonal | $x_t-x_{t-12}$ | Compara o mês atual com o mesmo mês do ano anterior e pode remover uma raiz unitária sazonal. |
| (e) | Primeira diferença + diferença sazonal | $(1-B)(1-B^{12})x_t$ | Remove simultaneamente componentes estocásticos regular e sazonal, quando ambos forem necessários. |
| (f) | Primeira diferença do log + diferença sazonal | $(1-B)(1-B^{12})\log(x_t)$ | Combina estabilização de escala, variação mensal e diferenciação sazonal. |
| (g) | Série sem tendência linear | $x_t-(\hat\beta_0+\hat\beta_1t)$ | Remove apenas uma tendência determinística estimada por mínimos quadrados. |

O operador $B$ é o operador de defasagem: $Bx_t=x_{t-1}$. A transformação mais complexa não é automaticamente a melhor. Diferenciações desnecessárias reduzem a amostra, dificultam a interpretação e podem criar autocorrelação negativa artificial.

In [10]:
def remover_tendencia(serie):
    s = pd.Series(serie).dropna().astype(float)
    tempo = np.arange(len(s), dtype=float)
    ajuste = OLS(s.to_numpy(), add_constant(tempo)).fit()
    return pd.Series(ajuste.resid, index=s.index)

def criar_transformacoes(serie, contagem=False):
    s = pd.Series(serie).astype(float)
    log_s = np.log1p(s) if contagem else np.log(s)
    prefixo_log = "log1p" if contagem else "log"
    return {
        "nivel": s,
        prefixo_log: log_s,
        "diff1": s.diff(1),
        f"{prefixo_log}_diff1": log_s.diff(1),
        "diff12": s.diff(12),
        "diff1_diff12": s.diff(1).diff(12),
        f"{prefixo_log}_diff1_diff12": log_s.diff(1).diff(12),
        "detrended": remover_tendencia(s)
    }

transformacoes = {
    "taxa_inadimplencia": criar_transformacoes(series["taxa_inadimplencia"]),
    "total_desastres": criar_transformacoes(series["total_desastres"], contagem=True)
}

rotulos_transformacoes = {
    "taxa_inadimplencia": {
        "nivel": "Série original — nível (sem transformação)",
        "log": "(a) Transformação logarítmica",
        "diff1": "(b) Primeira diferença",
        "log_diff1": "(c) Primeira diferença do log",
        "diff12": "(d) Diferença sazonal de ordem 12",
        "diff1_diff12": "(e) Primeira diferença + diferença sazonal",
        "log_diff1_diff12": "(f) Primeira diferença do log + diferença sazonal",
        "detrended": "(g) Remoção da tendência linear"
    },
    "total_desastres": {
        "nivel": "Série original — nível (sem transformação)",
        "log1p": "(a) Transformação log1p",
        "diff1": "(b) Primeira diferença",
        "log1p_diff1": "(c) Primeira diferença de log1p",
        "diff12": "(d) Diferença sazonal de ordem 12",
        "diff1_diff12": "(e) Primeira diferença + diferença sazonal",
        "log1p_diff1_diff12": "(f) Primeira diferença de log1p + diferença sazonal",
        "detrended": "(g) Remoção da tendência linear"
    }
}

for nome, dicionario in transformacoes.items():
    fig, axes = plt.subplots(4, 2, figsize=(15, 14))
    for ax, (transf, s) in zip(axes.flat, dicionario.items()):
        x = s.dropna()
        ax.plot(x.index, x, color=rotulos[nome][2], linewidth=1.05)
        ax.set_title(
            f"{rotulos_transformacoes[nome][transf]}\n"
            f"n = {len(x)} | média = {x.mean():.3f} | desvio-padrão = {x.std():.3f}",
            fontsize=10
        )
        ax.xaxis.set_major_locator(mdates.YearLocator(3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.suptitle(f"Transformações — {rotulos[nome][0]}", y=1.01, fontsize=15)
    fig.tight_layout()
    fig.savefig(FIGURES / f"05_{nome}_transformacoes.png", dpi=180, bbox_inches="tight")
    plt.show()

<Figure>

<Figure>

## 11. Comparação das transformações

Além de ADF e KPSS, a tabela registra sazonalidade residual, estabilidade de variância e autocorrelação nos lags 1 e 12. Para o sinal de diferenciação excessiva, considera-se uma autocorrelação negativa intensa no primeiro lag (`ACF(1) < -0,50`), sempre interpretada em conjunto com os demais diagnósticos.

In [11]:
linhas = []
for nome, dicionario in transformacoes.items():
    for transf, s in dicionario.items():
        x = s.dropna().astype(float)
        teste = executar_testes(x, nome, transf)
        grupos = [g.to_numpy() for _, g in x.groupby(x.index.month)]
        kw_p = kruskal(*grupos).pvalue
        acfs = acf(x, nlags=min(36, len(x) // 2 - 1), fft=False)
        var_diag = diagnosticar_variancia(x, f"{nome} — {transf}")
        linhas.append({
            **teste,
            "Identificação": rotulos_transformacoes[nome][transf],
            "Sazonalidade p-valor": kw_p,
            "Sazonalidade residual": "Sim" if kw_p < ALPHA else "Não detectada",
            "Razão de variâncias": var_diag["Razão var. (2ª/1ª)"],
            "Levene p-valor": var_diag["Levene p-valor"],
            "ACF lag 1": acfs[1],
            "ACF lag 12": acfs[12] if len(acfs) > 12 else np.nan,
            "Possível sobrediferenciação": bool(acfs[1] < -0.50)
        })

comparacao = pd.DataFrame(linhas)
colunas_exibicao = [
    "Série", "Identificação", "Transformação", "n", "ADF p-valor", "KPSS p-valor", "Conclusão conjunta",
    "Sazonalidade residual", "Razão de variâncias", "Levene p-valor",
    "ACF lag 1", "ACF lag 12", "Possível sobrediferenciação"
]
display(comparacao[colunas_exibicao])
comparacao.to_csv(TABLES / "diagnosticos_transformacoes_2013_2024.csv", index=False, sep=";", decimal=",")

,Série,Identificação,Transformação,n,ADF p-valor,KPSS p-valor,Conclusão conjunta,Sazonalidade residual,Razão de variâncias,Levene p-valor,ACF lag 1,ACF lag 12,Possível sobrediferenciação
0,taxa_inadimplencia,Série original — nível (sem transformação),nivel,144,0.2026,0.0139,Evidência de NÃO-estacionariedade,Não detectada,1.2297,0.3060,0.9520,0.3066,False
1,taxa_inadimplencia,(a) Transformação logarítmica,log,144,0.1958,0.0170,Evidência de NÃO-estacionariedade,Não detectada,1.7469,0.0393,0.9583,0.3175,False
2,taxa_inadimplencia,(b) Primeira diferença,diff1,143,0.0191,0.1000,Evidência de estacionariedade,Sim,1.3489,0.4337,0.1814,0.3222,False
3,taxa_inadimplencia,(c) Primeira diferença do log,log_diff1,143,0.0208,0.1000,Evidência de estacionariedade,Sim,1.6630,0.1229,0.1976,0.3252,False
4,taxa_inadimplencia,(d) Diferença sazonal de ordem 12,diff12,132,0.0645,0.1000,Resultado conflitante ou inconclusivo,Não detectada,2.2675,0.0001,0.9601,-0.1990,False
5,taxa_inadimplencia,(e) Primeira diferença + diferença sazonal,diff1_diff12,131,0.0023,0.1000,Evidência de estacionariedade,Não detectada,1.9239,0.0350,0.5130,-0.4376,False
6,taxa_inadimplencia,(f) Primeira diferença do log + diferença sazonal,log_diff1_diff12,131,0.0021,0.1000,Evidência de estacionariedade,Não detectada,2.3075,0.0084,0.5433,-0.4522,False
7,taxa_inadimplencia,(g) Remoção da tendência linear,detrended,144,0.1298,0.1000,Resultado conflitante ou inconclusivo,Não detectada,2.4289,0.0000,0.9523,0.1712,False
8,total_desastres,Série original — nível (sem transformação),nivel,144,0.0000,0.0100,Resultado conflitante ou inconclusivo,Não detectada,3.0249,0.0302,0.3795,0.2380,False
9,total_desastres,(a) Transformação log1p,log1p,144,0.5714,0.0100,Evidência de NÃO-estacionariedade,Não detectada,0.8724,0.4267,0.3733,0.3139,False


cell_18:10: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
cell_18:11: FutureWarning: kpss currently returns a plain tuple whose length and layout depends on the store argument (and which silently drops `lags` when store=True). In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning a KPSSResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
cell_18:10: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always

## 12. Escolha das séries estacionárias

A seleção combina os testes formais com a inspeção gráfica, a estabilidade da variância, a sazonalidade e a interpretação substantiva. Portanto, não se escolhe uma transformação simplesmente porque ela produz o menor p-valor.

### Taxa nacional de inadimplência

A série em nível apresenta forte persistência temporal e resultados convergentes para não estacionariedade: o ADF não rejeita a presença de raiz unitária, enquanto o KPSS rejeita a estacionariedade. A **primeira diferença**, identificada como transformação **(b)**, é a alternativa mínima que produz concordância favorável entre os dois testes. Ela possui ainda uma interpretação clara: aumento ou redução mensal da taxa de inadimplência em pontos percentuais.

Embora a primeira diferença do log também seja estacionária, o log não melhorou a estabilidade da variância da inadimplência. Por isso, acrescentaria complexidade sem ganho diagnóstico relevante.

### Quantidade total de desastres

Na série em nível, ADF e KPSS produzem resultados conflitantes. Além disso, a variância da segunda metade do período é aproximadamente três vezes a da primeira metade, indicando mudança importante de escala. A **primeira diferença de `log1p`**, transformação **(c)**, mantém a concordância entre ADF e KPSS e apresenta variância mais equilibrada entre as duas partes da amostra do que a primeira diferença em nível.

Essa transformação pode ser interpretada aproximadamente como a taxa de variação mensal da quantidade de desastres, com a vantagem de continuar matematicamente válida se uma contagem for igual a zero.

### Decisão final

- **inadimplência:** primeira diferença em nível. É a transformação mínima com concordância entre ADF e KPSS. O log não melhora a estabilidade da variância e reduziria a interpretação direta para variação em pontos percentuais;
- **desastres:** primeira diferença de `log1p`. A primeira diferença simples e a diferença de `log1p` passam em ADF e KPSS, mas `log1p` estabiliza melhor a forte mudança de escala da contagem ao longo do período. A série representa aproximadamente a variação percentual mensal na quantidade de ocorrências.

As duas séries ainda podem apresentar padrão mensal na variação. Isso é sazonalidade determinística residual, não evidência automática de raiz unitária sazonal. A diferença sazonal combinada foi descartada porque remove 12 observações adicionais e aumenta a autocorrelação negativa no lag 12. Na etapa de Granger, devem ser avaliadas dummies mensais e especificações sazonais como controles.

In [12]:
escolhas = {
    "taxa_inadimplencia": "diff1",
    "total_desastres": "log1p_diff1"
}

selecionadas = pd.DataFrame([
    comparacao.loc[(comparacao["Série"] == nome) & (comparacao["Transformação"] == transf)].iloc[0]
    for nome, transf in escolhas.items()
])
display(selecionadas[colunas_exibicao])

,Série,Identificação,Transformação,n,ADF p-valor,KPSS p-valor,Conclusão conjunta,Sazonalidade residual,Razão de variâncias,Levene p-valor,ACF lag 1,ACF lag 12,Possível sobrediferenciação
2,taxa_inadimplencia,(b) Primeira diferença,diff1,143,0.0191,0.1000,Evidência de estacionariedade,Sim,1.3489,0.4337,0.1814,0.3222,False
11,total_desastres,(c) Primeira diferença de log1p,log1p_diff1,143,0.0000,0.1000,Evidência de estacionariedade,Sim,0.6644,0.1191,-0.3833,0.2919,False


## 13. ACF e PACF

ACF e PACF orientam a identificação, mas não determinam sozinhas as ordens do modelo:

- ACF com decaimento lento sugere possível não estacionariedade;
- corte da ACF após `q` sugere componente MA(q);
- corte da PACF após `p` sugere componente AR(p);
- decaimentos graduais em ambas sugerem estrutura ARMA;
- picos em 12, 24 ou 36 indicam dependência sazonal;
- ACF(1) muito negativa após diferenciação pode indicar sobrediferenciação;
- ausência de picos significativos é compatível com ruído branco.

In [13]:
def safe_nlags(serie, max_lags=36):
    n = len(pd.Series(serie).dropna())
    return max(1, min(max_lags, n // 2 - 1))

def plotar_acf_pacf(serie, titulo, nome_arquivo):
    s = pd.Series(serie).dropna().astype(float)
    lags = safe_nlags(s)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
    plot_acf(s, lags=lags, ax=axes[0], zero=False, alpha=.05)
    plot_pacf(s, lags=lags, ax=axes[1], zero=False, alpha=.05, method="ywm")
    axes[0].set_title("ACF")
    axes[1].set_title("PACF")
    for ax in axes:
        for lag in (12, 24, 36):
            if lag <= lags:
                ax.axvline(lag, color="#b22222", linestyle="--", alpha=.45)
        ax.set_xlabel("Defasagem (meses)")
    fig.suptitle(titulo, y=1.02, fontsize=14)
    fig.tight_layout()
    fig.savefig(FIGURES / nome_arquivo, dpi=180, bbox_inches="tight")
    plt.show()

for nome in series.columns:
    plotar_acf_pacf(series[nome], f"Série em nível — {rotulos[nome][0]}", f"05_{nome}_nivel_acf_pacf.png")
    transf = escolhas[nome]
    plotar_acf_pacf(
        transformacoes[nome][transf],
        f"Transformação selecionada: {rotulos_transformacoes[nome][transf]} — {rotulos[nome][0]}",
        f"05_{nome}_{transf}_acf_pacf.png"
    )

<Figure>

<Figure>

<Figure>

<Figure>

## 14. Bases finais para causalidade de Granger

A análise principal utiliza as transformações mínimas selecionadas:

$$
\Delta I_t = I_t-I_{t-1}
$$

e

$$
\Delta\log(1+D_t)=\log(1+D_t)-\log(1+D_{t-1}).
$$

Como ambas usam uma primeira diferença, janeiro de 2013 é perdido e a base principal começa em fevereiro de 2013.

Para a análise de sensibilidade do notebook 06, também é construída uma segunda base com diferença sazonal aplicada **depois** da primeira diferença:

$$
\Delta_{12}\Delta I_t=\Delta I_t-\Delta I_{t-12}
$$

e

$$
\Delta_{12}\Delta\log(1+D_t)
=\Delta\log(1+D_t)-\Delta\log(1+D_{t-12}).
$$

Essa transformação compara a variação mensal atual com a variação mensal observada no mesmo mês do ano anterior. Ela pode remover dependência sazonal remanescente, mas custa 12 observações adicionais e pode introduzir autocorrelação negativa no lag 12. Por isso, será tratada como **robustez**, e não como substituta automática da especificação principal com dummies mensais.

In [14]:
granger = pd.concat({
    "inad_diff1": transformacoes["taxa_inadimplencia"]["diff1"],
    "desastres_log1p_diff1": transformacoes["total_desastres"]["log1p_diff1"]
}, axis=1).dropna()
granger.index.name = "data"

if not granger.index.is_monotonic_increasing or granger.index.has_duplicates:
    raise ValueError("O índice final não está ordenado ou possui duplicidades.")
if granger.isna().any().any() or np.isinf(granger.to_numpy(dtype="float64")).any():
    raise ValueError("A base final contém nulos ou infinitos.")
if granger.nunique().le(1).any():
    raise ValueError("A base final contém série constante.")

auditoria_final = pd.DataFrame([
    executar_testes(granger[coluna], coluna, "selecionada") for coluna in granger.columns
])
display(Markdown("### Base principal: primeiras diferenças"))
display(auditoria_final)
if not ((auditoria_final["ADF rejeita H0"]) & (~auditoria_final["KPSS rejeita H0"])).all():
    raise ValueError("Ao menos uma série principal deixou de satisfazer conjuntamente ADF e KPSS após o alinhamento.")

granger.reset_index().to_csv(
    PROCESSED / "series_estacionarias_granger_2013_2024.csv",
    index=False, sep=";", decimal="," 
)
try:
    granger.reset_index().to_parquet(
        PROCESSED / "series_estacionarias_granger_2013_2024.parquet", index=False
    )
    parquet_status = "exportado"
except (ImportError, ModuleNotFoundError):
    parquet_status = "não exportado: instale pyarrow"

resumo_final = pd.DataFrame([
    {
        "variavel_original": "taxa_inadimplencia",
        "tipo": "taxa percentual",
        "transformacao_selecionada": "primeira diferença",
        "formula": "x_t - x_(t-1)",
        "coluna_granger": "inad_diff1",
        "n": len(granger),
        "periodo_final": f"{granger.index.min():%Y-%m} a {granger.index.max():%Y-%m}",
        "interpretacao": "variação mensal da taxa de inadimplência, em pontos percentuais"
    },
    {
        "variavel_original": "total_desastres",
        "tipo": "contagem",
        "transformacao_selecionada": "primeira diferença de log1p",
        "formula": "log(1+x_t) - log(1+x_(t-1))",
        "coluna_granger": "desastres_log1p_diff1",
        "n": len(granger),
        "periodo_final": f"{granger.index.min():%Y-%m} a {granger.index.max():%Y-%m}",
        "interpretacao": "variação logarítmica mensal da quantidade de desastres"
    }
])
resumo_final.to_csv(TABLES / "resumo_estacionariedade_2013_2024.csv", index=False, sep=";", decimal=",")
auditoria_final.to_csv(TABLES / "testes_adf_kpss_series_granger_2013_2024.csv", index=False, sep=";", decimal=",")
resumo_final.to_csv(TABLES / "metadados_series_granger_2013_2024.csv", index=False, sep=";", decimal=",")

# Base de robustez com primeira diferença seguida de diferença sazonal de ordem 12.
granger_sazonal = pd.concat({
    "inad_diff1_diff12": transformacoes["taxa_inadimplencia"]["diff1_diff12"],
    "desastres_log1p_diff1_diff12": transformacoes["total_desastres"]["log1p_diff1_diff12"]
}, axis=1).dropna()
granger_sazonal.index.name = "data"

if not granger_sazonal.index.is_monotonic_increasing or granger_sazonal.index.has_duplicates:
    raise ValueError("O índice da base sazonal não está ordenado ou possui duplicidades.")
if granger_sazonal.isna().any().any() or np.isinf(granger_sazonal.to_numpy(dtype="float64")).any():
    raise ValueError("A base sazonal contém nulos ou infinitos.")
if granger_sazonal.nunique().le(1).any():
    raise ValueError("A base sazonal contém série constante.")

auditoria_sazonal = pd.DataFrame([
    executar_testes(granger_sazonal[coluna], coluna, "diff1 + diff12")
    for coluna in granger_sazonal.columns
])

acf_sazonal = []
for coluna in granger_sazonal.columns:
    x = granger_sazonal[coluna].astype(float)
    valores_acf = acf(x, nlags=12, fft=False)
    limite_aprox = 1.96 / np.sqrt(len(x))
    acf_sazonal.append({
        "Série": coluna,
        "n": len(x),
        "ACF(1)": valores_acf[1],
        "ACF(12)": valores_acf[12],
        "Limite aproximado 95%": limite_aprox,
        "ACF(12) significativa": bool(abs(valores_acf[12]) > limite_aprox),
    })
acf_sazonal = pd.DataFrame(acf_sazonal)

granger_sazonal.reset_index().to_csv(
    PROCESSED / "series_estacionarias_granger_sazonal_2013_2024.csv",
    index=False, sep=";", decimal=","
)
try:
    granger_sazonal.reset_index().to_parquet(
        PROCESSED / "series_estacionarias_granger_sazonal_2013_2024.parquet", index=False
    )
    parquet_sazonal_status = "exportado"
except (ImportError, ModuleNotFoundError):
    parquet_sazonal_status = "não exportado: instale pyarrow"

resumo_sazonal = pd.DataFrame([
    {
        "variavel_original": "taxa_inadimplencia",
        "transformacao": "primeira diferença + diferença sazonal",
        "formula": "Delta12 Delta I_t",
        "coluna_granger": "inad_diff1_diff12",
        "n": len(granger_sazonal),
        "periodo_final": f"{granger_sazonal.index.min():%Y-%m} a {granger_sazonal.index.max():%Y-%m}",
        "interpretacao": "diferença entre a variação mensal atual e a do mesmo mês do ano anterior"
    },
    {
        "variavel_original": "total_desastres",
        "transformacao": "primeira diferença de log1p + diferença sazonal",
        "formula": "Delta12 Delta log(1+D_t)",
        "coluna_granger": "desastres_log1p_diff1_diff12",
        "n": len(granger_sazonal),
        "periodo_final": f"{granger_sazonal.index.min():%Y-%m} a {granger_sazonal.index.max():%Y-%m}",
        "interpretacao": "diferença entre a variação logarítmica mensal atual e a do mesmo mês do ano anterior"
    }
])

auditoria_sazonal.to_csv(TABLES / "testes_adf_kpss_series_granger_sazonal_2013_2024.csv", index=False, sep=";", decimal=",")
resumo_sazonal.to_csv(TABLES / "metadados_series_granger_sazonal_2013_2024.csv", index=False, sep=";", decimal=",")
acf_sazonal.to_csv(TABLES / "acf_series_granger_sazonal_2013_2024.csv", index=False, sep=";", decimal=",")

display(Markdown("### Base de robustez: primeira diferença + diferença sazonal"))
display(auditoria_sazonal)
display(acf_sazonal)
display(resumo_sazonal)
display(granger_sazonal.head())

print("Base principal:")
print("  Observações originais:", len(series))
print("  Observações finais:", len(granger))
print("  Observações perdidas:", len(series) - len(granger))
print("  Período:", granger.index.min().strftime("%Y-%m"), "a", granger.index.max().strftime("%Y-%m"))
print("  Parquet:", parquet_status)
print("Base de robustez sazonal:")
print("  Observações finais:", len(granger_sazonal))
print("  Observações perdidas em relação à série original:", len(series) - len(granger_sazonal))
print("  Período:", granger_sazonal.index.min().strftime("%Y-%m"), "a", granger_sazonal.index.max().strftime("%Y-%m"))
print("  Parquet:", parquet_sazonal_status)

Base principal:
  Observações originais: 144
  Observações finais: 143
  Observações perdidas: 1
  Período: 2013-02 a 2024-12
  Parquet: exportado
Base de robustez sazonal:
  Observações finais: 131
  Observações perdidas em relação à série original: 13
  Período: 2014-02 a 2024-12
  Parquet: exportado


### Base principal: primeiras diferenças

,Série,Transformação,n,ADF estatística,ADF p-valor,ADF lags,KPSS estatística,KPSS p-valor,KPSS lags,ADF rejeita H0,KPSS rejeita H0,Conclusão conjunta
0,inad_diff1,selecionada,143,-3.2154,0.0191,12,0.1611,0.1000,6,True,False,Evidência de estacionariedade
1,desastres_log1p_diff1,selecionada,143,-6.9653,0.0000,10,0.0558,0.1000,14,True,False,Evidência de estacionariedade


### Base de robustez: primeira diferença + diferença sazonal

,Série,Transformação,n,ADF estatística,ADF p-valor,ADF lags,KPSS estatística,KPSS p-valor,KPSS lags,ADF rejeita H0,KPSS rejeita H0,Conclusão conjunta
0,inad_diff1_diff12,diff1 + diff12,131,-3.8603,0.0023,12,0.0869,0.1000,5,True,False,Evidência de estacionariedade
1,desastres_log1p_diff1_diff12,diff1 + diff12,131,-5.4902,0.0000,13,0.3290,0.1000,54,True,False,Evidência de estacionariedade


,Série,n,ACF(1),ACF(12),Limite aproximado 95%,ACF(12) significativa
0,inad_diff1_diff12,131,0.5130,-0.4376,0.1712,True
1,desastres_log1p_diff1_diff12,131,-0.2927,-0.2740,0.1712,True


,variavel_original,transformacao,formula,coluna_granger,n,periodo_final,interpretacao
0,taxa_inadimplencia,primeira diferença + diferença sazonal,Delta12 Delta I_t,inad_diff1_diff12,131,2014-02 a 2024-12,diferença entre a variação mensal atual e a do...
1,total_desastres,primeira diferença de log1p + diferença sazonal,Delta12 Delta log(1+D_t),desastres_log1p_diff1_diff12,131,2014-02 a 2024-12,diferença entre a variação logarítmica mensal ...


,inad_diff1_diff12,desastres_log1p_diff1_diff12
data,,
2014-02-01,0.0125,0.1265
2014-03-01,0.0228,1.5371
2014-04-01,0.1328,-0.8127
2014-05-01,0.0426,-0.1797
2014-06-01,0.1454,1.4555


cell_18:10: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
cell_18:11: FutureWarning: kpss currently returns a plain tuple whose length and layout depends on the store argument (and which silently drops `lags` when store=True). In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning a KPSSResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
cell_18:10: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always

## 15. Verificação de prontidão para a causalidade de Granger

A causalidade de Granger verifica se valores passados de uma variável acrescentam capacidade preditiva para outra variável, depois de considerados os próprios valores passados da variável explicada. Assim, “causar no sentido de Granger” significa **precedência temporal com ganho preditivo**, e não causalidade econômica, física ou social em sentido estrito.

Antes da estimação, a base deve satisfazer quatro condições operacionais:

1. as séries utilizadas precisam ser estacionárias na especificação adotada;
2. as observações devem estar na mesma frequência, alinhadas e sem lacunas artificiais;
3. deve existir quantidade suficiente de observações para estimar as defasagens;
4. componentes determinísticos remanescentes, como sazonalidade mensal, devem ser controlados no modelo.

O diagnóstico abaixo separa condições obrigatórias de pontos de atenção. A indicação de “pronta com controle sazonal” significa que a análise pode avançar, mas o próximo notebook deve incorporar dummies mensais — e comparar com a base de robustez sazonal agora exportada — para evitar que um padrão de calendário seja confundido com ganho preditivo.

In [15]:
def linha_prontidao(criterio, resultado, status, implicacao):
    return {"Critério": criterio, "Resultado": resultado, "Status": status, "Implicação para Granger": implicacao}

estacionarias = bool(
    auditoria_final["ADF rejeita H0"].all()
    and (~auditoria_final["KPSS rejeita H0"]).all()
)
indice_valido = bool(granger.index.is_monotonic_increasing and not granger.index.has_duplicates)
valores_validos = bool(not granger.isna().any().any() and np.isfinite(granger.to_numpy(dtype="float64")).all())
amostra_suficiente = bool(len(granger) >= 100)

p_sazonal_inad = float(comparacao.loc[
    (comparacao["Série"] == "taxa_inadimplencia") & (comparacao["Transformação"] == "diff1"),
    "Sazonalidade p-valor"
].iloc[0])
p_sazonal_desastres = float(comparacao.loc[
    (comparacao["Série"] == "total_desastres") & (comparacao["Transformação"] == "log1p_diff1"),
    "Sazonalidade p-valor"
].iloc[0])
sazonalidade_residual = bool((p_sazonal_inad < ALPHA) or (p_sazonal_desastres < ALPHA))

prontidao = pd.DataFrame([
    linha_prontidao(
        "Estacionariedade conjunta (ADF e KPSS)",
        "As duas séries apresentam ADF < 5% e KPSS ≥ 5%",
        "Atendido" if estacionarias else "Não atendido",
        "Permite usar o teste convencional nas séries transformadas."
    ),
    linha_prontidao(
        "Alinhamento temporal",
        f"{len(granger)} meses, de {granger.index.min():%m/%Y} a {granger.index.max():%m/%Y}",
        "Atendido" if indice_valido else "Não atendido",
        "As duas variáveis se referem aos mesmos meses e estão em ordem cronológica."
    ),
    linha_prontidao(
        "Ausência de nulos e infinitos",
        "Nenhum valor inválido" if valores_validos else "Há valores inválidos",
        "Atendido" if valores_validos else "Não atendido",
        "Não é necessário preencher ou interpolar observações."
    ),
    linha_prontidao(
        "Tamanho da amostra",
        f"n = {len(granger)}",
        "Atendido" if amostra_suficiente else "Atenção",
        "É possível comparar modelos com até 12 defasagens, privilegiando ordens parcimoniosas."
    ),
    linha_prontidao(
        "Sazonalidade determinística residual",
        f"p(inad.) = {p_sazonal_inad:.4g}; p(desastres) = {p_sazonal_desastres:.4g}",
        "Atenção — controlar" if sazonalidade_residual else "Não detectada",
        "Incluir dummies mensais no VAR/Granger e verificar os resíduos."
    )
])

condicoes_obrigatorias = estacionarias and indice_valido and valores_validos and amostra_suficiente
decisao_granger = "PRONTA, COM CONTROLE SAZONAL" if condicoes_obrigatorias else "AINDA NÃO PRONTA"
display(prontidao)
display(Markdown(f"### Decisão: base **{decisao_granger}** para a análise de Granger"))
prontidao.to_csv(TABLES / "diagnostico_prontidao_granger_2013_2024.csv", index=False, sep=";", decimal=",")

,Critério,Resultado,Status,Implicação para Granger
0,Estacionariedade conjunta (ADF e KPSS),As duas séries apresentam ADF < 5% e KPSS ≥ 5%,Atendido,Permite usar o teste convencional nas séries t...
1,Alinhamento temporal,"143 meses, de 02/2013 a 12/2024",Atendido,As duas variáveis se referem aos mesmos meses ...
2,Ausência de nulos e infinitos,Nenhum valor inválido,Atendido,Não é necessário preencher ou interpolar obser...
3,Tamanho da amostra,n = 143,Atendido,É possível comparar modelos com até 12 defasag...
4,Sazonalidade determinística residual,p(inad.) = 2.877e-08; p(desastres) = 0.00294,Atenção — controlar,Incluir dummies mensais no VAR/Granger e verif...


### Decisão: base **PRONTA, COM CONTROLE SAZONAL** para a análise de Granger

### Recomendações para o notebook `06_causalidade_granger.ipynb`

- estimar as duas direções: desastres → inadimplência e inadimplência → desastres;
- comparar ordens mensais por **AIC e BIC**, reportando os testes nas duas ordens quando houver divergência;
- incluir onze dummies mensais como regressoras determinísticas, deixando janeiro como referência;
- comparar a especificação sazonal principal com um modelo sem dummies;
- repetir a seleção por AIC/BIC e os testes com `series_estacionarias_granger_sazonal_2013_2024.*` como robustez;
- verificar estabilidade, autocorrelação residual e, especificamente, o valor da ACF dos resíduos no lag 12;
- apresentar o teste conjunto dos coeficientes defasados, e não interpretar um coeficiente isolado como prova de causalidade;
- interpretar os resultados considerando a redução da amostra, possível sobrediferenciação, variáveis omitidas e o caráter agregado nacional dos dados.

## 16. Conclusão gerada a partir dos resultados

In [16]:
def pvalor(nome, transf, teste):
    coluna = f"{teste} p-valor"
    return float(comparacao.loc[(comparacao["Série"] == nome) & (comparacao["Transformação"] == transf), coluna].iloc[0])

texto = f"""
### Síntese dos diagnósticos

A taxa de inadimplência em nível apresentou ADF com p-valor {pvalor('taxa_inadimplencia','nivel','ADF'):.4f} e KPSS com p-valor {pvalor('taxa_inadimplencia','nivel','KPSS'):.4f}, combinação favorável à não estacionariedade. A primeira diferença apresentou ADF com p-valor {pvalor('taxa_inadimplencia','diff1','ADF'):.4f} e KPSS com p-valor {pvalor('taxa_inadimplencia','diff1','KPSS'):.4f}; por isso, foi selecionada como a transformação mínima adequada. O log não foi mantido porque não trouxe ganho de estabilidade suficiente e a diferença em nível possui interpretação direta em pontos percentuais.

Para o total de desastres, os testes em nível foram conflitantes: o ADF rejeitou raiz unitária, mas o KPSS rejeitou estacionariedade. A primeira diferença de `log1p` apresentou ADF com p-valor {pvalor('total_desastres','log1p_diff1','ADF'):.4g} e KPSS com p-valor {pvalor('total_desastres','log1p_diff1','KPSS'):.4f}. Ela foi escolhida por conciliar estacionariedade e maior estabilidade da escala da contagem. A transformação usa `log1p(x)=log(1+x)`, sendo válida mesmo se houver meses com zero ocorrências.

Após o alinhamento, a base contém **{len(granger)} observações**, de **{granger.index.min():%m/%Y} a {granger.index.max():%m/%Y}**, sem nulos, infinitos ou duplicidades. As condições obrigatórias de estacionariedade, alinhamento e integridade foram atendidas. Portanto, a base está **pronta para a análise de Granger com controle sazonal**. O próximo notebook deverá selecionar a ordem de defasagem e incluir dummies mensais, pois as transformações escolhidas ainda apresentam diferenças sistemáticas entre meses do ano. A base sazonalmente diferenciada também foi exportada, com **{len(granger_sazonal)} observações**, de **{granger_sazonal.index.min():%m/%Y} a {granger_sazonal.index.max():%m/%Y}**, para ser usada exclusivamente como análise de sensibilidade no notebook 06. A perda adicional de 12 meses e o risco de sobrediferenciação devem ser considerados na interpretação.

Esses testes possuem limitações de potência e podem ser afetados por quebras estruturais, eventos extremos e especificação das defasagens. Estacionariedade não garante causalidade verdadeira: Granger avalia precedência temporal e ganho preditivo condicional. A próxima etapa será `06_causalidade_granger.ipynb`.
"""
display(Markdown(texto))


### Síntese dos diagnósticos

A taxa de inadimplência em nível apresentou ADF com p-valor 0.2026 e KPSS com p-valor 0.0139, combinação favorável à não estacionariedade. A primeira diferença apresentou ADF com p-valor 0.0191 e KPSS com p-valor 0.1000; por isso, foi selecionada como a transformação mínima adequada. O log não foi mantido porque não trouxe ganho de estabilidade suficiente e a diferença em nível possui interpretação direta em pontos percentuais.

Para o total de desastres, os testes em nível foram conflitantes: o ADF rejeitou raiz unitária, mas o KPSS rejeitou estacionariedade. A primeira diferença de `log1p` apresentou ADF com p-valor 8.949e-10 e KPSS com p-valor 0.1000. Ela foi escolhida por conciliar estacionariedade e maior estabilidade da escala da contagem. A transformação usa `log1p(x)=log(1+x)`, sendo válida mesmo se houver meses com zero ocorrências.

Após o alinhamento, a base contém **143 observações**, de **02/2013 a 12/2024**, sem nulos, infinitos ou duplicidades. As condições obrigatórias de estacionariedade, alinhamento e integridade foram atendidas. Portanto, a base está **pronta para a análise de Granger com controle sazonal**. O próximo notebook deverá selecionar a ordem de defasagem e incluir dummies mensais, pois as transformações escolhidas ainda apresentam diferenças sistemáticas entre meses do ano. A base sazonalmente diferenciada também foi exportada, com **131 observações**, de **02/2014 a 12/2024**, para ser usada exclusivamente como análise de sensibilidade no notebook 06. A perda adicional de 12 meses e o risco de sobrediferenciação devem ser considerados na interpretação.

Esses testes possuem limitações de potência e podem ser afetados por quebras estruturais, eventos extremos e especificação das defasagens. Estacionariedade não garante causalidade verdadeira: Granger avalia precedência temporal e ganho preditivo condicional. A próxima etapa será `06_causalidade_granger.ipynb`.


## 17. Arquivos produzidos

### Bases para Granger

- `data/processed/series_estacionarias_granger_2013_2024.csv` e `.parquet`: primeiras diferenças usadas na análise principal;
- `data/processed/series_estacionarias_granger_sazonal_2013_2024.csv` e `.parquet`: primeiras diferenças seguidas de diferença sazonal, usadas como robustez.

### Tabelas e figuras

- `outputs/tables/diagnosticos_transformacoes_2013_2024.csv`;
- tabelas de testes ADF/KPSS e metadados das duas bases de Granger;
- `outputs/tables/acf_series_granger_sazonal_2013_2024.csv`;
- `outputs/tables/diagnostico_prontidao_granger_2013_2024.csv`;
- figuras com prefixo `05_` em `outputs/figures/`.

As séries com diferença sazonal são fornecidas para sensibilidade. Elas não substituem automaticamente a especificação principal com dummies mensais.